# AIS YOLO Pose Train

`port_pair` training bbox 1개와 port0/port1의 4개 corner keypoint를 학습합니다.

Assumption:
- YOLO pose label format: `class x_center y_center width height kpt0_x kpt0_y ... kpt7_x kpt7_y`
- keypoint order: `port0_top_left, port0_top_right, port0_bottom_right, port0_bottom_left, port1_top_left, port1_top_right, port1_bottom_right, port1_bottom_left`
- visibility까지 저장한 라벨이면 `KEYPOINT_DIMS = 3`으로 바꿉니다.
- assumption: insertion orientation is already corrected.
- with that canonical orientation, port0 is the left port in the image and port1 is the right port.
- corner 순서가 뒤집히면 학습 결과가 바로 나빠지므로 annotation 쪽 순서와 이 노트북의 `KEYPOINT_NAMES`를 반드시 맞춥니다.

In [1]:
from pathlib import Path
import random

import yaml


def find_src_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pixi.toml").exists() and (path / "ais").exists():
            return path
        nested_src = path / "ws_aic" / "src"
        if (nested_src / "pixi.toml").exists() and (nested_src / "ais").exists():
            return nested_src
    raise RuntimeError("ws_aic/src root를 찾지 못했습니다. 노트북을 ws_aic/src 아래에서 실행하세요.")


SRC_ROOT = find_src_root(Path.cwd().resolve())
WS_ROOT = SRC_ROOT.parent

# annotation/collection 쪽에서 생성한 dataset root로 바꾸세요.
DATASET_DIR = WS_ROOT / "data" / "yolo" / "approach" / "SFP"
DATA_YAML = DATASET_DIR / "data.yaml"

KEYPOINT_DIMS = 2  # x,y only. visibility가 있으면 3.
KEYPOINT_NAMES = [
    "port0_top_left",
    "port0_top_right",
    "port0_bottom_right",
    "port0_bottom_left",
    "port1_top_left",
    "port1_top_right",
    "port1_bottom_right",
    "port1_bottom_left",
]

CLASS_NAMES = {0: "port_pair"}
IMAGE_EXTS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

print(f"SRC_ROOT: {SRC_ROOT}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"DATA_YAML: {DATA_YAML}")

SRC_ROOT: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/src
DATASET_DIR: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP
DATA_YAML: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP/data.yaml


## Dataset yaml

Ultralytics pose dataset 설정을 생성합니다. 라벨이 이미 준비된 뒤 실행하세요.

In [2]:
dataset_cfg = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "kpt_shape": [len(KEYPOINT_NAMES), KEYPOINT_DIMS],
    # corner/port index swap을 확정하기 전까지 horizontal flip은 train에서 끕니다.
    "flip_idx": list(range(len(KEYPOINT_NAMES))),
    "names": CLASS_NAMES,
    "kpt_names": {0: KEYPOINT_NAMES},
}

DATASET_DIR.mkdir(parents=True, exist_ok=True)
DATA_YAML.write_text(yaml.safe_dump(dataset_cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

path: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP
train: images/train
val: images/val
kpt_shape:
- 8
- 2
flip_idx:
- 0
- 1
- 2
- 3
- 4
- 5
- 6
- 7
names:
  0: port_pair
kpt_names:
  0:
  - port0_top_left
  - port0_top_right
  - port0_bottom_right
  - port0_bottom_left
  - port1_top_left
  - port1_top_right
  - port1_bottom_right
  - port1_bottom_left



## Label check

학습 전에 이미지/라벨 개수와 YOLO pose 라벨 컬럼 수를 확인합니다.

In [3]:
def resolve_split_dir(cfg: dict, split: str) -> Path:
    split_path = Path(cfg[split]).expanduser()
    if split_path.is_absolute():
        return split_path

    root = Path(cfg.get("path", DATA_YAML.parent)).expanduser()
    if not root.is_absolute():
        root = DATA_YAML.parent / root
    return (root / split_path).resolve()


def label_path_for_image(image_path: Path) -> Path:
    parts = list(image_path.parts)
    if "images" in parts:
        parts[parts.index("images")] = "labels"
        return Path(*parts).with_suffix(".txt")
    return image_path.with_suffix(".txt")


def image_path_for_label(label_path: Path) -> Path | None:
    parts = list(label_path.parts)
    if "labels" in parts:
        parts[parts.index("labels")] = "images"
        stem_path = Path(*parts).with_suffix("")
        for ext in IMAGE_EXTS:
            candidate = stem_path.with_suffix(ext)
            if candidate.exists():
                return candidate
    return None


def iter_images(image_dir: Path) -> list[Path]:
    if not image_dir.exists():
        return []
    return sorted(p for p in image_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def validate_label_file(label_path: Path, expected_cols: int) -> list[str]:
    errors = []
    for line_no, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        stripped = line.strip()
        if not stripped:
            continue
        parts = stripped.split()
        if len(parts) != expected_cols:
            errors.append(f"{label_path}:{line_no}: expected {expected_cols} columns, got {len(parts)}")
            continue
        try:
            class_id = int(float(parts[0]))
            values = [float(v) for v in parts[1:]]
        except ValueError:
            errors.append(f"{label_path}:{line_no}: non-numeric value")
            continue
        if class_id not in CLASS_NAMES:
            errors.append(f"{label_path}:{line_no}: unknown class_id {class_id}")
        bbox = values[:4]
        if any(v < 0.0 or v > 1.0 for v in bbox):
            errors.append(f"{label_path}:{line_no}: bbox values must be normalized 0..1")
        kpts = values[4:]
        for i in range(0, len(kpts), KEYPOINT_DIMS):
            xy = kpts[i:i + 2]
            if any(v < 0.0 or v > 1.0 for v in xy):
                errors.append(f"{label_path}:{line_no}: keypoint xy values must be normalized 0..1")
            if KEYPOINT_DIMS == 3 and not (0.0 <= kpts[i + 2] <= 2.0):
                errors.append(f"{label_path}:{line_no}: visibility must be 0, 1, or 2")
    return errors


# Keep this cell robust even if the notebook kernel still has an old DATASET_DIR.
DATASET_DIR = WS_ROOT / "data" / "yolo" / "approach" / "SFP"
DATA_YAML = DATASET_DIR / "data.yaml"
print(f"Checking dataset: {DATASET_DIR}")

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
expected_cols = 5 + len(KEYPOINT_NAMES) * KEYPOINT_DIMS
all_errors = []

for split in ["train", "val"]:
    image_dir = resolve_split_dir(cfg, split)
    images = iter_images(image_dir)
    labels = [label_path_for_image(p) for p in images]
    missing_labels = [p for p in labels if not p.exists()]
    existing_labels = [p for p in labels if p.exists()]

    label_dir = Path(str(image_dir).replace("/images/", "/labels/"))
    orphan_labels = []
    if label_dir.exists():
        orphan_labels = [p for p in label_dir.rglob("*.txt") if image_path_for_label(p) is None]

    print(f"{split}: images={len(images)}, labels={len(existing_labels)}, missing_labels={len(missing_labels)}, orphan_labels={len(orphan_labels)}")
    if not images:
        all_errors.append(f"{split}: no images under {image_dir}")
    all_errors.extend(f"missing label: {p}" for p in missing_labels[:20])
    all_errors.extend(f"orphan label: {p}" for p in orphan_labels[:20])
    for label_path in existing_labels[:]:
        all_errors.extend(validate_label_file(label_path, expected_cols))

if all_errors:
    preview = "\n".join(all_errors[:50])
    raise ValueError(f"Label check failed with {len(all_errors)} errors.\n{preview}")

print(f"OK: expected label columns = {expected_cols}")

Checking dataset: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP
train: images=834, labels=834, missing_labels=0, orphan_labels=0
val: images=421, labels=421, missing_labels=0, orphan_labels=0
OK: expected label columns = 21


## Train

`MODEL_NAME`은 필요하면 `yolo11n-pose.pt`, `yolo11s-pose.pt`, `yolov8s-pose.pt` 등으로 바꿉니다.

In [4]:
from ultralytics import YOLO

MODEL_NAME = "yolo11s-pose.pt"
OUTPUT_DIR = WS_ROOT / "model" / "ais_yolo" / "approach" / "SFP"

EPOCHS = 100
IMGSZ = 640
BATCH = 16
DEVICE = 0  # CPU는 "cpu"
WORKERS = 8
PATIENCE = 20

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(OUTPUT_DIR.parent),
    name=OUTPUT_DIR.name,
    patience=PATIENCE,
    save=True,
    save_period=10,
    plots=True,
    device=DEVICE,
    workers=WORKERS,
    fliplr=0.0,
    flipud=0.0,
)

best_pt = OUTPUT_DIR / "weights" / "best.pt"
print(f"best.pt: {best_pt}")

New https://pypi.org/project/ultralytics/8.4.50 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.40 🚀 Python-3.10.0 torch-2.10.0+cu130 CUDA:0 (NVIDIA GB10, 122566MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-pose.pt, momen

/home/vsc/LLM_TUNE/structure-stability/venv/venv-stability/lib/python3.10/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


YOLO11s-pose summary: 197 layers, 9,715,843 parameters, 9,715,827 gradients, 22.5 GFLOPs

Transferred 505/541 items from pretrained weights
Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3729.6±1090.4 MB/s, size: 114.8 KB)
train: Scanning /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP/labels/train... 834 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 834/834 2.1Kit/s 0.4s0.0ss
train: New cache created: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP/labels/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1898.9±1549.3 MB/s, size: 100.2 KB)
val: Scanning /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP/labels/val... 421 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 421/421 1.4Kit/s 0.3s<0.2s
val: New cache created: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP/labels/val.cache
optimizer: 'opti

## Resume or validate only

필요할 때만 아래 셀을 실행합니다.

In [ ]:
# last.pt에서 완전 재개할 때 사용합니다.
# resume_pt = OUTPUT_DIR / "weights" / "last.pt"
# model = YOLO(str(resume_pt))
# results = model.train(resume=True)

In [ ]:
# 학습 없이 검증만 할 때 사용합니다.
# model = YOLO(str(OUTPUT_DIR / "weights" / "best.pt"))
# metrics = model.val(data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, device=DEVICE)
# print(metrics)

In [5]:
# 샘플 이미지 예측 확인용입니다.
import cv2
import matplotlib.pyplot as plt

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
sample_images = iter_images(resolve_split_dir(cfg, "val")) or iter_images(resolve_split_dir(cfg, "train"))
sample = random.choice(sample_images)
model = YOLO(str(OUTPUT_DIR / "weights" / "best.pt"))
pred = model.predict(
    source=str(sample),
    imgsz=IMGSZ,
    device=DEVICE,
    save=True,
    project=str(OUTPUT_DIR.parent),
    name=f"{OUTPUT_DIR.name}_predict",
)
annotated = pred[0].plot()
plt.figure(figsize=(8, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(sample.name)
print(sample)


image 1/1 /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP/images/val/sfp_mount_ds003_ep00003_right.jpg: 576x640 1 port_pair, 17.1ms
Speed: 1.3ms preprocess, 17.1ms inference, 0.8ms postprocess per image at shape (1, 3, 576, 640)
Results saved to /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/model/ais_yolo/approach/SFP_predict
/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/yolo/approach/SFP/images/val/sfp_mount_ds003_ep00003_right.jpg
